# 2.1. Créer une V2 du notebook avec tuning FLAML
Ce notebook duplique le scénario d'entraînement et ajoute un tuning des hyperparamètres avec FLAML sur une tâche de régression.

## Convention commune
Ce notebook utilise le même modèle logique que le reste du module : `prediction_velos_horaires`. Les runs de cette version sont enregistrés dans l'expérience `ds_prediction_velos_v2_flaml`.

In [ ]:
%pip install -q flaml
from flaml import AutoML
import mlflow
import mlflow.sklearn
import pandas as pd
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
source_path = "Files/comptage-velo-donnees-compteurs.csv"
df = spark.read.option("header", True).option("sep", ";").csv(source_path)
prepared_df = (
    df
    .withColumn("date_time", F.to_timestamp("Date et heure de comptage"))
    .withColumn("jour", F.to_date("date_time"))
    .withColumn("heure", F.hour("date_time"))
    .withColumn("station", F.col("Nom du site de comptage"))
    .withColumn("nb_velos", F.col("Comptage horaire").cast("double"))
    .withColumn("jour_semaine", F.dayofweek("jour"))
    .withColumn("mois", F.month("jour"))
    .select("station", "jour_semaine", "mois", "heure", "nb_velos")
    .dropna()
)
display(prepared_df.limit(10))

## Préparer les données pour FLAML
Cette étape convertit les données préparées en pandas et crée les matrices d'entraînement et de test.

In [ ]:
pdf = prepared_df.toPandas()
pdf = pd.get_dummies(pdf, columns=["station"], dtype=float)
X = pdf.drop(columns=["nb_velos"])
y = pdf["nb_velos"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
automl = AutoML()
settings = {
    "task": "regression",
    "metric": "rmse",
    "time_budget": 120,
    "log_file_name": "flaml.log"
}
automl.fit(X_train=X_train, y_train=y_train, **settings)
preds = automl.predict(X_test)
rmse = mean_squared_error(y_test, preds, squared=False)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print({"best_estimator": automl.best_estimator, "best_config": automl.best_config, "rmse": rmse, "mae": mae, "r2": r2})

In [ ]:
experiment_name = "ds_prediction_velos_v2_flaml"
run_name = "v2_flaml_tuned"
registered_model_name = "prediction_velos_horaires"

mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_name=run_name):
    mlflow.log_param("registered_model_name", registered_model_name)
    mlflow.log_param("best_estimator", automl.best_estimator)
    mlflow.log_params(automl.best_config)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(automl.model.estimator, artifact_path="model")

## À retenir
FLAML permet d'obtenir rapidement une version optimisée de la baseline, avec une recherche automatisée d'hyperparamètres et une journalisation distincte dans `ds_prediction_velos_v2_flaml`.
## Exercice
Augmenter `time_budget`, relancer le tuning, puis comparer le meilleur estimateur et le `rmse` à la version précédente.